#  Notebook 4: Geospatial Analytics — Delivery & Churn by Brazilian State

In [1]:
import pandas as pd
import numpy as np
import folium
from folium import Choropleth, GeoJsonTooltip
import json, requests, os
from sqlalchemy import create_engine
warnings._filters_mutated if False else None
import warnings; warnings.filterwarnings('ignore')

DB_PATH = os.path.join('..', 'data', 'olist.db')
engine = create_engine(f'sqlite:///{DB_PATH}')
os.makedirs('../outputs/maps', exist_ok=True)


## 1. State-Level Delivery & Revenue Stats

In [2]:
state_query = '''
SELECT
    c.customer_state,
    COUNT(DISTINCT o.order_id)           AS total_orders,
    ROUND(AVG(CASE
        WHEN o.order_delivered_customer_date > o.order_estimated_delivery_date
        THEN JULIANDAY(o.order_delivered_customer_date) - JULIANDAY(o.order_estimated_delivery_date)
        ELSE 0 END), 2)                  AS avg_delay_days,
    ROUND(AVG(CASE
        WHEN o.order_delivered_customer_date <= o.order_estimated_delivery_date
        THEN 1.0 ELSE 0.0 END), 4)      AS on_time_rate,
    ROUND(SUM(op.payment_value), 2)     AS total_revenue,
    ROUND(AVG(r.review_score), 2)       AS avg_review_score
FROM orders o
JOIN customers c         ON o.customer_id = c.customer_id
JOIN order_payments op   ON o.order_id = op.order_id
LEFT JOIN order_reviews r ON o.order_id = r.order_id
WHERE o.order_status = 'delivered'
  AND o.order_delivered_customer_date IS NOT NULL
GROUP BY c.customer_state
'''
state_df = pd.read_sql(state_query, engine)
print(state_df.sort_values('total_revenue', ascending=False).head(10).to_string(index=False))


customer_state  total_orders  avg_delay_days  on_time_rate  total_revenue  avg_review_score
            SP         40493            0.41        0.9421     5798662.08              4.25
            RJ         12350            1.73        0.8654     2063882.53              3.97
            MG         11354            0.41        0.9447     1826872.47              4.19
            RS          5344            0.67        0.9294      866012.87              4.18
            PR          4923            0.37        0.9501      785305.81              4.24
            SC          3546            0.76        0.9029      597148.45              4.13
            BA          3256            1.55        0.8589      593541.16              3.93
            DF          2080            0.45        0.9306      348624.23              4.13
            GO          1957            0.77        0.9182      337747.38              4.09
            ES          1995            1.30        0.8787      318491.23       

## 2. Merge Churn Data

In [3]:
churn_df = pd.read_csv('../outputs/churn_predictions.csv')
churn_state = churn_df.groupby('customer_state').agg(
    churn_rate      =('churn','mean'),
    avg_churn_prob  =('churn_prob','mean'),
    revenue_at_risk =('revenue_at_risk','sum')
).reset_index()

state_full = state_df.merge(churn_state, on='customer_state', how='left')
state_full.to_csv('../outputs/state_summary.csv', index=False)
print(f"State summary built: {len(state_full)} states")
print(state_full[['customer_state','churn_rate','revenue_at_risk']].sort_values(
    'revenue_at_risk', ascending=False).head(10).to_string(index=False))


State summary built: 27 states
customer_state  churn_rate  revenue_at_risk
            SP    0.566271       183.689462
            RJ    0.630931        65.379473
            MG    0.612824        57.871491
            RS    0.629117        27.433473
            PR    0.596994        24.876842
            SC    0.633390        18.916411
            BA    0.607801        18.802140
            DF    0.577404        11.043685
            GO    0.622381        10.699129
            ES    0.625063        10.089135


## 3. Download Brazil GeoJSON

In [4]:
GEOJSON_URL = "https://raw.githubusercontent.com/codeforamerica/click_that_hood/master/public/data/brazil-states.json"
GEOJSON_PATH = "../outputs/brazil_states.geojson"

try:
    resp = requests.get(GEOJSON_URL, timeout=15)
    resp.raise_for_status()
    with open(GEOJSON_PATH, 'w') as f:
        f.write(resp.text)
    geojson = resp.json()
    # Inspect property keys
    props = geojson['features'][0]['properties']
except Exception as e:
    print(f"Download failed: {e}. Maps will use CircleMarker fallback.")
    geojson = None


Download failed: 404 Client Error: Not Found for url: https://raw.githubusercontent.com/codeforamerica/click_that_hood/master/public/data/brazil-states.json. Maps will use CircleMarker fallback.


## 4. Choropleth Map 1 — Churn Rate by State

In [5]:
def make_choropleth(data, col, title, fill_color, geojson, key_on='feature.properties.sigla'):
    m = folium.Map(location=[-14.2, -51.9], zoom_start=4, tiles='CartoDB dark_matter')
    if geojson:
        Choropleth(
            geo_data=geojson,
            data=data, columns=['customer_state', col],
            key_on=key_on,
            fill_color=fill_color, fill_opacity=0.75, line_opacity=0.4,
            legend_name=title, nan_fill_color='#2d2d2d'
        ).add_to(m)
        # Tooltip
        tooltip_data = data.set_index('customer_state')[col].to_dict()
        style_fn   = lambda x: {'fillOpacity': 0, 'weight': 0}
        highlight  = lambda x: {'weight': 2, 'color': 'white'}
        folium.GeoJson(
            geojson,
            style_function=style_fn,
            highlight_function=highlight,
            tooltip=folium.GeoJsonTooltip(
                fields=['name','sigla'], aliases=['State','Code'])
        ).add_to(m)
    else:
        # CircleMarker fallback
        STATE_COORDS = {
            'AC':(-9.0,-70.8),'AL':(-9.5,-36.8),'AM':(-3.4,-65.8),'AP':(1.4,-51.8),
            'BA':(-12.9,-38.5),'CE':(-5.2,-39.3),'DF':(-15.8,-47.9),'ES':(-19.2,-40.3),
            'GO':(-15.8,-49.8),'MA':(-4.9,-45.3),'MG':(-18.5,-44.6),'MS':(-20.5,-54.6),
            'MT':(-12.6,-55.9),'PA':(-3.4,-52.0),'PB':(-7.2,-36.8),'PE':(-8.8,-36.9),
            'PI':(-7.7,-42.7),'PR':(-24.9,-51.6),'RJ':(-22.9,-43.4),'RN':(-5.8,-36.5),
            'RO':(-11.5,-63.6),'RR':(1.9,-61.2),'RS':(-30.0,-53.2),'SC':(-27.2,-50.2),
            'SE':(-10.5,-37.1),'SP':(-23.5,-46.6),'TO':(-10.2,-48.3),
        }
        for _, row in data.iterrows():
            coords = STATE_COORDS.get(row['customer_state'], (-14.2,-51.9))
            val = row.get(col, 0)
            folium.CircleMarker(
                location=coords, radius=8+val*20,
                color='white', fill=True, fill_opacity=0.7,
                tooltip=f"{row['customer_state']}: {val:.3f}"
            ).add_to(m)
    return m

if geojson or True:
    m1 = make_choropleth(state_full, 'churn_rate', 'Churn Rate', 'YlOrRd', geojson)
    m1.save('../outputs/maps/churn_rate_map.html')
    display(m1)


## 5. Choropleth Map 2 — Avg Delivery Delay

In [6]:
m2 = make_choropleth(state_full, 'avg_delay_days', 'Avg Delivery Delay (days)', 'RdPu', geojson)
m2.save('../outputs/maps/delivery_delay_map.html')
display(m2)


## 6. Choropleth Map 3 — Revenue at Risk

In [7]:
m3 = make_choropleth(state_full, 'revenue_at_risk', 'Revenue at Risk (BRL)', 'YlGn', geojson)
m3.save('../outputs/maps/revenue_at_risk_map.html')
display(m3)


## 7. State Summary Table

In [8]:
print("\n=== Top 10 States by Revenue at Risk ===")
top10 = state_full.sort_values('revenue_at_risk', ascending=False).head(10)
print(top10[['customer_state','total_orders','avg_delay_days',
             'on_time_rate','churn_rate','revenue_at_risk']].to_string(index=False))



=== Top 10 States by Revenue at Risk ===
customer_state  total_orders  avg_delay_days  on_time_rate  churn_rate  revenue_at_risk
            SP         40493            0.41        0.9421    0.566271       183.689462
            RJ         12350            1.73        0.8654    0.630931        65.379473
            MG         11354            0.41        0.9447    0.612824        57.871491
            RS          5344            0.67        0.9294    0.629117        27.433473
            PR          4923            0.37        0.9501    0.596994        24.876842
            SC          3546            0.76        0.9029    0.633390        18.916411
            BA          3256            1.55        0.8589    0.607801        18.802140
            DF          2080            0.45        0.9306    0.577404        11.043685
            GO          1957            0.77        0.9182    0.622381        10.699129
            ES          1995            1.30        0.8787    0.625063        